# **Oxford Flowers Clustering Notebook**

In this notebook, we:
1. Load the validation and test splits of the Oxford Flower Dataset, merging them.
2. Compute the CLIP embeddings with the `ClipEmbedder`.
3. Cluster these images into 102 clusters (the number of classes).
4. Inspect the clusters: the first images of a few clusters and a 3D t-SNE projection of the embeddings.
5. Compute RI, ARI, NMI, and interpret the results.
6. Repeat the clustering on the similarity matrix instead of the embeddings.

---

## **1. Setup and Load Data**

In [ ]:
from collections.abc import Sequence

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib.axes import Axes
from sklearn.cluster import SpectralClustering
from sklearn.manifold import TSNE
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    rand_score,
)

from pyvisim.datasets import OxfordFlowerDataset
from pyvisim.distance import cosine_similarity
from pyvisim.neural_networks import ClipEmbedder
from pyvisim.typing import FloatNumpyArray, IntNumpyArray

### Helpers

All helpers used in this notebook are defined here. The first cell clusters the features and scores the result against the ground-truth classes.

In [ ]:
RANDOM_STATE = 42


def cluster_spectrally(features: FloatNumpyArray, n_clusters: int) -> IntNumpyArray:
    """
    Assign every row of ``features`` to one of ``n_clusters`` spectral clusters.

    :param features: Array of shape (N, D) with one feature vector per image
    :param n_clusters: Number of clusters to form
    :return: Cluster id of every image, shape (N,)
    """
    model = SpectralClustering(
        n_clusters=n_clusters, affinity="nearest_neighbors", random_state=RANDOM_STATE
    )
    return np.asarray(model.fit_predict(features))


def score_clustering(
    true_labels: IntNumpyArray, cluster_labels: IntNumpyArray
) -> dict[str, float]:
    """
    Compare a clustering against the ground-truth classes.

    :param true_labels: Ground-truth class of every image, shape (N,)
    :param cluster_labels: Cluster id of every image, shape (N,)
    :return: Rand index, adjusted Rand index and normalized mutual information
    """
    return {
        "RI": rand_score(true_labels, cluster_labels),
        "ARI": adjusted_rand_score(true_labels, cluster_labels),
        "NMI": normalized_mutual_info_score(true_labels, cluster_labels),
    }


def print_scores(scores: dict[str, float], heading: str) -> None:
    """
    Print clustering scores under a heading.

    :param scores: Scores keyed by their name, as returned by :func:`score_clustering`
    :param heading: Line printed above the scores
    """
    print(heading)
    for name, value in scores.items():
        print(f"{name}: {value:.4f}")

The next cell shows the first images that were assigned to a cluster, one row per cluster. Every image is titled with its ground-truth class, so a row with mixed classes reveals an impure cluster at a glance.

In [ ]:
def show_cluster_samples(
    dataset: OxfordFlowerDataset,
    cluster_labels: IntNumpyArray,
    cluster_ids: Sequence[int],
    samples_per_cluster: int,
) -> None:
    """
    Show the first images assigned to each cluster, one row per cluster.

    :param dataset: Dataset the clustering was computed on
    :param cluster_labels: Cluster id of every image in ``dataset``, shape (N,)
    :param cluster_ids: Clusters to show, in row order
    :param samples_per_cluster: Number of images per row
    """
    _, axes = plt.subplots(
        len(cluster_ids),
        samples_per_cluster,
        figsize=(3 * samples_per_cluster, 3 * len(cluster_ids)),
        squeeze=False,
    )
    for row, cluster_id in zip(axes.tolist(), cluster_ids, strict=True):
        members = np.flatnonzero(cluster_labels == cluster_id)[:samples_per_cluster]
        draw_cluster_row(row, dataset, members, cluster_id)
    plt.tight_layout()
    plt.show()


def draw_cluster_row(
    row: Sequence[Axes],
    dataset: OxfordFlowerDataset,
    members: IntNumpyArray,
    cluster_id: int,
) -> None:
    """
    Draw the images of one cluster onto a row of axes.

    Axes left over when the cluster has fewer members than the row has axes
    are hidden.

    :param row: Axes of the row, one per image
    :param dataset: Dataset the images are read from
    :param members: Indices into ``dataset`` of the images to draw
    :param cluster_id: Cluster the images belong to, shown as the row label
    """
    for axis, index in zip(row, members, strict=False):
        image, label, _ = dataset[index]
        axis.imshow(image)
        axis.set_title(f"Class {label}")
    for axis in row[len(members) :]:
        axis.set_visible(False)
    for axis in row:
        axis.set_xticks([])
        axis.set_yticks([])
    row[0].set_ylabel(f"Cluster {cluster_id}", fontsize=12)

The last two helpers plot data: a 3D t-SNE projection of the embeddings, in which every cluster gets its own colour, and an annotated heatmap of a similarity matrix.

In [ ]:
def project_to_3d(features: FloatNumpyArray) -> FloatNumpyArray:
    """
    Project feature vectors to three dimensions with t-SNE.

    :param features: Array of shape (N, D)
    :return: Array of shape (N, 3)
    """
    return TSNE(n_components=3, perplexity=40, random_state=RANDOM_STATE).fit_transform(features)


def plot_clusters_3d(
    points: FloatNumpyArray,
    cluster_labels: IntNumpyArray,
    title: str,
    colormap_name: str = "nipy_spectral",
) -> None:
    """
    Scatter 3D points, colouring the points of a cluster with the same colour.

    :param points: Array of shape (N, 3)
    :param cluster_labels: Cluster id of every point, shape (N,)
    :param title: Title of the plot
    :param colormap_name: Colormap the cluster palette is sampled from
    """
    unique_labels, dense_indices = np.unique(cluster_labels, return_inverse=True)
    cluster_count = unique_labels.size

    # Skip the colormap extremes: they are black/near-white in spectral maps.
    palette = plt.get_cmap(colormap_name)(np.linspace(0.05, 0.95, cluster_count))
    stride = np.concatenate(
        [np.arange(0, cluster_count, 2), np.arange(1, cluster_count, 2)]
    )
    colours = palette[stride][dense_indices]
    colours[cluster_labels < 0] = (0.65, 0.65, 0.65, 0.35)

    axis = plt.figure(figsize=(10, 10)).add_subplot(projection="3d")
    axis.set_box_aspect(None, zoom=0.85)
    axis.scatter(points[:, 0], points[:, 1], points[:, 2], c=colours, s=10)
    axis.set_title(title)
    axis.set_xlabel("t-SNE 1")
    axis.set_ylabel("t-SNE 2")
    axis.set_zlabel("t-SNE 3")
    plt.tight_layout()
    plt.show()


def show_similarity_heatmap(matrix: FloatNumpyArray, title: str) -> None:
    """
    Draw an annotated heatmap of a similarity matrix.

    :param matrix: Square array of pairwise similarities
    :param title: Title of the plot
    """
    plt.figure(figsize=(8, 7))
    sns.heatmap(
        matrix,
        annot=True,
        fmt=".2f",
        cmap="Blues",
        cbar_kws={"label": "Cosine similarity"},
    )
    plt.title(title)
    plt.xlabel("Image index")
    plt.ylabel("Image index")
    plt.show()

### Load the validation and test datasets.

In [ ]:
dataset = OxfordFlowerDataset(purpose=["validation", "test"])
true_labels = np.array(dataset.labels)
print("Number of images in the dataset:", len(dataset))

## **2. Compute CLIP embeddings**

### Define the CLIP embedder

In [ ]:
clip_embedder = ClipEmbedder(variant="ViT-B/32", pretrained="openai")

### Compute the CLIP embeddings for both the validation and test splits

In [ ]:
clip_embeddings = clip_embedder.embed(image for image, *_ in dataset)

## **3. Cluster into 102 Clusters**

`102` is the number of classes in the Oxford Flowers dataset. We want to see how well the clustering algorithm can cluster the images into these classes.


In [ ]:
NUM_CLASSES = 102
cluster_labels = cluster_spectrally(clip_embeddings, n_clusters=NUM_CLASSES)

### Inspect the clusters

Before looking at any score, let's look at the clusters themselves. The grid below shows the first 5 images of each of the first 5 clusters. Ideally, all images in a row share the same class.

In [ ]:
show_cluster_samples(
    dataset, cluster_labels, cluster_ids=range(5), samples_per_cluster=5
)

### Visualize the embeddings in 3D

t-SNE is used to project the 512-dimensional CLIP embeddings onto three dimensions.

In [ ]:
projected_embeddings = project_to_3d(clip_embeddings)
plot_clusters_3d(
    projected_embeddings,
    cluster_labels,
    title=f"t-SNE projection of the CLIP embeddings, {NUM_CLASSES} spectral clusters",
)

### Compute RI, ARI, NMI

The Rand index (RI) is the fraction of image pairs on which the clustering and the ground truth agree. The adjusted Rand index (ARI) corrects it for chance agreement, and the normalized mutual information (NMI) measures how much knowing the cluster of an image tells us about its class.

In [ ]:
scores = score_clustering(true_labels, cluster_labels)
heading = f"Spectral clustering of the embeddings into {NUM_CLASSES} clusters:"
print_scores(scores, heading)

## **4. Cluster on the Similarity Matrix**

Now, instead of using the embeddings themselves, we will use the `similarity matrix` of the embeddings to cluster the images. So each row in this matrix will represent the similarity of an image to all other images in the dataset (hence, all diagonal elements will be 1).

In [ ]:
similarity_matrix = cosine_similarity(clip_embeddings, clip_embeddings)
show_similarity_heatmap(
    similarity_matrix[:10, :10],
    title="Similarity matrix of the dataset, first 10 images",
)

In [ ]:
similarity_cluster_labels = cluster_spectrally(
    similarity_matrix, n_clusters=NUM_CLASSES
)
similarity_scores = score_clustering(true_labels, similarity_cluster_labels)
heading = f"Spectral clustering of the similarity matrix into {NUM_CLASSES} clusters:"
print_scores(similarity_scores, heading)

## **5. Conclusion**

We've demonstrated:
- How to cluster images directly on CLIP embeddings.
- How to inspect the clusters visually, through sample images and a 3D t-SNE projection.
- How to compute RI, ARI and NMI for objective evaluation.

For CLIP, clustering on the embeddings themselves performs significantly better than clustering on the similarity matrix.